# Introduction to LLM APIs

## From HTTP APIs to Large Language Models

In the previous lesson, you learned how to **consume APIs from Python** and how to build simple APIs with **Flask** and **FastAPI**.

Today we will use exactly the same ideas to communicate with a **Large Language Model (LLM)**.

### Learning goals

By the end of this notebook, you should be able to:

- distinguish between an **LLM**, its **tokenizer**, and an **API/provider** used to access it;
- explore models on **Hugging Face** and identify criteria for choosing one;
- send an LLM request manually using `requests`;
- inspect the request payload and the raw JSON response;
- wrap an LLM call inside a reusable Python function;
- understand the roles of `system`, `user`, and `assistant` messages;
- maintain simple conversation history;
- experiment with generation parameters such as `temperature`;
- use an LLM to help **interpret** results produced with pandas;
- recognize why LLM answers need appropriate **context** and may hallucinate.

> **Main idea:** an LLM API is still an API. We send structured data to an endpoint and receive structured data back.

## 1. Model, tokenizer, and API provider

An LLM application involves several pieces that are easy to confuse:

```text
Text
  ↓
Tokenizer
  ↓
Tokens / token IDs
  ↓
LLM
  ↓
Generated tokens
  ↓
Tokenizer / decoder
  ↓
Text
```

A useful mental model is therefore:

> **LLM + compatible tokenizer**

Different model families can use different tokenizers. In many libraries and hosted services the tokenizer is handled automatically, so we do not normally call it ourselves.

There is another independent question: **where does the model run?**

For example:

```text
Meta
  ↓ develops
Llama model + tokenizer
  ↓ hosted by
Groq
  ↓ exposes
HTTP API
  ↓ called from
Python
```

The **model** and the **API provider** are not the same thing.

The same or similar open model may be available through different inference providers, with different prices, limits, latency, hardware, and APIs.

### Check for understanding

For each item below, decide whether it is primarily a **model/model family**, a **model hub**, or an **inference/API provider**:

- Llama
- Hugging Face
- Groq
- Qwen

Discuss your answer with the person next to you before continuing.

## 2. Exploring models on Hugging Face

Open the **[Hugging Face Models](https://huggingface.co/models)** page in your browser.

Search for a few text-generation / instruction models, for example models from the **Llama**, **Qwen**, or **Phi** families.

Do not choose a model only because it is popular. Inspect its **model card**.

Things worth checking include:

- **Task / use case** — chat, instruction following, coding, classification, etc.
- **Model size** — number of parameters and hardware requirements.
- **Context window** — how much context the model can process.
- **Language support**.
- **License** — what are you allowed to do with the model?
- **Performance / evaluations** — useful, but benchmark results should not be treated as the only criterion.
- **Deployment requirements** — can you realistically run it locally, or do you need an inference provider?

> There is no universally "best" LLM. The appropriate model depends on the problem and constraints.

### Mini activity

Choose **two models** that could be used for a chatbot and compare them. Write down at least **three reasons** why you might choose one over the other.

## 3. Setup: API credentials

For the practical part we will use the **Groq API** and a hosted language model.

Create an API key in your Groq account and store it in a local `.env` file:

```text
GROQ_API_KEY="your_key_here"
```

Never upload API keys to GitHub.

Your `.gitignore` should contain:

```text
.env
```

We will retrieve the credential from the environment rather than hard-coding it in the notebook.


In [ ]:
# If needed, install the packages in your environment first:
# pip install requests python-dotenv pandas

import os
import requests
import pandas as pd

from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("GROQ_API_KEY")

if API_KEY is None:
    raise ValueError("GROQ_API_KEY was not found. Check your .env file.")

print("API key loaded successfully.")

## 4. Discovering available models

Before sending our first prompt, we will ask the API which models are currently available.

This is useful because **external APIs change over time**. A model that exists today may be renamed, restricted, or removed in the future.

Groq provides a `GET /models` endpoint that returns its active models. Notice the distinction:

```text
GET  /models             → discover available resources
POST /chat/completions   → send a prompt to a selected model
```

The Models API does **not** tell us directly whether a model belongs to the Free Plan. For this lesson, we therefore keep a short list of preferred models that we have previously selected as suitable for the activity, and compare that list with the models currently returned by the API.


In [ ]:
BASE_URL = "https://api.groq.com/openai/v1"
MODELS_URL = f"{BASE_URL}/models"
CHAT_URL = f"{BASE_URL}/chat/completions"

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

response = requests.get(MODELS_URL, headers=headers, timeout=30)

print("Status code:", response.status_code)
response.raise_for_status()

models = response.json()["data"]
available_models = [model["id"] for model in models]

for model in available_models:
    print(model)


### Selecting a model safely

We do **not** simply use the first model returned by the API. The list may contain speech-to-text models, safety models, or other systems that are not appropriate for this chat exercise.

Instead, we maintain a small ordered list of preferred chat models. The first preferred model that is also currently available will be selected.

> **Maintenance note:** the preferred list represents models chosen for this lesson and known to be suitable for the Free Plan when the notebook was reviewed. Provider plans and catalogues can change, so this short list may occasionally need updating.


In [ ]:
preferred_models = [
    "openai/gpt-oss-20b",
    "openai/gpt-oss-120b",
]

MODEL = next(
    (
        model
        for model in preferred_models
        if model in available_models
    ),
    None,
)

if MODEL is None:
    raise RuntimeError(
        "None of the preferred chat models are currently available. "
        "Check the provider's current model catalogue."
    )

print("Selected model:", MODEL)


### Check for understanding

Why is this more robust than writing a model ID directly in every API call?

And why would this be a bad idea?

```python
MODEL = available_models[0]
```

Think about the different kinds of models an inference provider may expose.


## 5. Our first LLM request — the manual way

We will deliberately start **"the hard way"** using `requests`.

This should look familiar from the previous lesson. We already have authentication, an endpoint, and a model selected dynamically. Now we can send a `POST` request containing a conversation.


In [ ]:
payload = {
    "model": MODEL,
    "messages": [
        {
            "role": "user",
            "content": "Explain what an API is in one short paragraph."
        }
    ]
}

response = requests.post(
    CHAT_URL,
    headers=headers, # Our credentials
    json=payload, # What we want
    timeout=30,
)

print("Status code:", response.status_code)
print(response.text)

### What did we just send?

The request has the same ingredients as other HTTP APIs:

- an **endpoint** (`URL`);
- **headers**, including authentication;
- a JSON **payload**;
- an HTTP method: `POST`.

The unusual part is mainly the structure expected by the LLM endpoint.

Before extracting the generated text, inspect the **entire response**.

In [ ]:
response.raise_for_status()
result = response.json()
result

### Mini activity: investigate the JSON

Without running the next cell yet, inspect `result` and find:

1. the model that generated the response;
2. the generated message;
3. the role assigned to that message;
4. any information about token usage.

Then determine the sequence of dictionary keys/list indexes needed to extract only the generated text.

In [ ]:
# Complete this expression before looking at the solution.
# answer = result[...][...][...][...]

# Uncomment after trying it yourself:
# answer = result["choices"][0]["message"]["content"]
# print(answer)

## 6. From a raw request to a reusable function

Repeating all the HTTP code every time would be inconvenient.

Let's wrap the request in a simple Python function.

In [ ]:
def ask_llm(prompt, model=MODEL, temperature=0.7):
    payload = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": temperature,
    }

    response = requests.post(
        CHAT_URL,
        headers=headers,
        json=payload,
        timeout=30,
    )
    response.raise_for_status()

    data = response.json()
    return data["choices"][0]["message"]["content"]

In [ ]:
answer = ask_llm("What is the difference between a list and a tuple in Python?")
print(answer)

### Check for understanding

What has the function hidden from us?

The HTTP request has **not disappeared**. The function simply abstracts away repeated steps:

```text
prompt
  ↓
ask_llm()
  ↓
HTTP POST request
  ↓
Groq API
  ↓
LLM
  ↓
JSON response
  ↓
extract generated content
  ↓
Python string
```

This is the same reason client libraries and SDKs exist: they provide a more convenient interface over lower-level API operations.

## 6. Messages and roles

Chat-oriented LLM APIs usually represent a conversation as a **list of messages**.

A message contains a `role` and `content`.

Common roles include:

- `system` — instructions describing how the model should behave;
- `user` — input from the user;
- `assistant` — previous responses from the model.

Let's make our function accept a complete message history.

In [ ]:
def chat(messages, model=MODEL, temperature=0.7):
    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
    }

    response = requests.post(
        CHAT_URL,
        headers=headers,
        json=payload,
        timeout=30,
    )
    response.raise_for_status()

    data = response.json()
    return data["choices"][0]["message"]["content"]

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a Data Analytics instructor. Explain concepts clearly and concisely."
    },
    {
        "role": "user",
        "content": "What is a primary key?"
    }
]

answer = chat(messages)
print(answer)

## 8. Conversation memory

An important detail: the model does not magically remember previous API calls.

If we want a later request to use an earlier exchange, we send the relevant conversation history again.

```text
Request 1: [system, user]
                    ↓
                 answer

Request 2: [system, user, assistant, new user]
```

In other words, **our application manages the conversation history**.

In [ ]:
conversation = [
    {
        "role": "system",
        "content": "You are a Data Analytics instructor. Keep answers concise."
    },
    {
        "role": "user",
        "content": "Explain a SQL LEFT JOIN using customers and orders."
    }
]

first_answer = chat(conversation)
print(first_answer)

# Store the assistant response in the history.
conversation.append({"role": "assistant", "content": first_answer})

# Add a follow-up question that depends on the previous answer.
conversation.append({
    "role": "user",
    "content": "Now explain how the result would differ with an INNER JOIN."
})

second_answer = chat(conversation)
print("FOLLOW-UP:")
print(second_answer)

### Mini activity: remove the memory

Create a new request containing **only** this message:

```text
Now explain how the result would differ with an INNER JOIN.
```

Compare the result with the previous response.

**Question:** why is the second prompt less well grounded when the previous conversation is not included?

In [ ]:
# Your experiment here

## 9. Temperature and generation

LLMs generate text by predicting tokens. Generation parameters influence how the model selects among possible continuations.

One common parameter is `temperature`.

As a simplified intuition:

- **lower temperature** → generally more conservative / predictable generation;
- **higher temperature** → generally more varied generation.

Temperature does **not** make a model more knowledgeable and should not be interpreted as a simple "creativity percentage".

Let's compare outputs.

In [ ]:
prompt = "Suggest a short title for a dashboard about online retail sales. Return only the title."

for temperature in [0.0, 0.5, 1.0]:
    print(f"Temperature = {temperature}")
    print(ask_llm(prompt, temperature=temperature))
    print("-" * 50)

### Mini activity

Run the previous cell several times.

- Are the answers always identical?
- Does increasing temperature guarantee a better answer?
- For which Data Analytics tasks might you prefer a lower temperature?

> Exact behavior depends on the model and provider. Generation can also be affected by other sampling parameters.

## 9. LLMs need context

Consider this question:

> **What was the total revenue of my company last year?**

The model has not been given your company's private business data.

What should happen if we ask anyway?

In [ ]:
print(ask_llm("What was the total revenue of my company last year?"))

A responsible answer should explain that the information is missing. But an LLM can sometimes produce plausible-sounding statements that are unsupported or incorrect.

This is one form of what is commonly called an **LLM hallucination**.

A key principle is therefore:

> **If the answer depends on information the model does not have, provide the relevant context.**

But context is not the same as blindly sending every available piece of data. We should decide what information is relevant and appropriate to share.

## 11. Data Analytics use case: pandas calculates, the LLM interprets

An LLM should not replace the tools that already solve deterministic analytical tasks well.

For example, let pandas calculate metrics first.

In [ ]:
sales = pd.DataFrame({
    "product": ["Laptop", "Monitor", "Keyboard", "Mouse"],
    "sales_eur": [125000, 68000, 31000, 27000],
    "units_sold": [500, 850, 1550, 1800],
    "returns": [34, 71, 22, 18],
})

sales["return_rate"] = sales["returns"] / sales["units_sold"]
sales["revenue_share"] = sales["sales_eur"] / sales["sales_eur"].sum()

sales

Now Python has calculated the metrics. We can give those results to an LLM and ask it to **interpret and communicate** them.

Notice the separation of responsibilities:

```text
Raw data
   ↓
pandas
   ↓
calculated metrics
   ↓
LLM + instructions + context
   ↓
written interpretation
```

> **pandas calculates. The LLM helps interpret or communicate.**

In [ ]:
summary = sales.to_csv(index=False)

prompt = f"""
You are a data analyst preparing a short note for a sales manager.

The metrics below have already been calculated in Python.

{summary}

Identify the two most important business insights supported by these data.
For each insight, explicitly mention the metric that supports it.
Do not invent causes that are not present in the data.
"""

print(ask_llm(prompt, temperature=0.2))

### Critical thinking

Read the response carefully.

Ask yourself:

- Are all numerical claims supported by the DataFrame?
- Did the model confuse **sales revenue** with **units sold**?
- Did it infer a cause that is not present in the data?
- Are the "most important" insights objectively determined, or is that partly a judgment?

LLM output should be **reviewed**, especially when it is used to communicate analytical results.

## 11. Your turn: improve the analytical prompt

Modify the previous prompt so that the model produces an **executive summary with exactly three bullet points**.

Requirements:

- each bullet must contain at least one value from the data;
- one bullet must discuss revenue;
- one bullet must discuss return rate;
- the model must distinguish **observation** from **possible explanation**;
- it must not invent information that is not present in the table.

Compare your result with a classmate's prompt.

In [ ]:
# Write your improved prompt here.

## 13. Optional: what does an SDK change?

We started with `requests` intentionally because it exposes what is happening.

A provider SDK can make the same workflow shorter and more convenient. Conceptually, however, it still performs API operations for us.

```text
SDK method
   ↓
build HTTP request
   ↓
API endpoint
   ↓
parse response
   ↓
Python object
```

For this introductory lesson, understanding the HTTP request is more important than learning a provider-specific SDK.

# Takeaways

1. An **LLM/model family** is not the same thing as the **provider/API** used to run it.
2. An LLM works together with a **compatible tokenizer**; hosted services often handle tokenization automatically.
3. Hugging Face model cards help us investigate model purpose, size, license, limitations, and other characteristics.
4. Calling an LLM API uses the same HTTP concepts as other APIs: endpoint, authentication, headers, payload, status code, and JSON response.
5. Chat APIs represent conversations as structured **messages with roles**.
6. API-based conversation memory usually means **sending relevant history again**.
7. Parameters such as `temperature` influence generation behavior.
8. LLMs need appropriate **context** and can generate unsupported statements.
9. In Data Analytics, deterministic tools such as pandas should perform calculations; LLMs can help interpret and communicate the resulting information.
10. API keys belong in environment variables / `.env` files — **never in notebooks committed to GitHub**.

---

## Final challenge

Build a small Python function that receives a pandas DataFrame and a business question and returns a short LLM-generated interpretation.

Before sending anything to the API, decide:

- What information from the DataFrame is actually needed?
- Should Python calculate any metrics first?
- What instructions will prevent unsupported conclusions?
- What output format would be most useful to the user?

The goal is not to make the LLM "do the analysis for you". The goal is to design a sensible pipeline in which **Python and the LLM each do the job they are best suited for**.


In [ ]:
import jupyterlab
jupyterlab.__version__